# Workflows

An agent chooses its next step. A workflow follows steps we chose in advance.

In this notebook, we will learn four workflow shapes:

- routing;
- prompt chaining;
- parallel work; and
- orchestrator with workers.

A fixed workflow is often simpler when the path is already known.

In [ ]:
import concurrent.futures as futures
import json
import os
import time
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

from shoplab import world
from shoplab.llm import LEDGER, llm, parse_json_loose

tickets = world.load_tickets()
orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}

## 1. Routing

A router sends each ticket to the right kind of handler.

We will use three lanes:

- simple_refund for an easy case;
- needs_policy when rules must be checked; and
- escalate_review when a person should inspect risk.

In [ ]:
LANES = ("simple_refund", "needs_policy", "escalate_review")

def route(ticket):
    prompt = f"""Choose one lane for this return.
simple_refund: clear and routine
needs_policy: a rule must be checked
escalate_review: fraud or unusual risk

Reply with only the lane name.

{ticket['reason_text']}"""

    answer = llm(prompt, model=MODEL).strip().lower()
    for lane in LANES:
        if lane in answer:
            return lane
    return "needs_policy"

for ticket in tickets["dev"][:4]:
    print(ticket["ticket_id"], "->", route(ticket))

Each lane gets only the amount of work it needs. An easy ticket uses one model call. A policy ticket gets the full tool agent. A risky ticket stops for human review.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

SYSTEM = (
    "Handle this Larkspur return. Look up the order, customer, and policy. "
    "Then call finish with decision, policy_id, and refund_usd."
)

def ticket_text(ticket):
    return json.dumps({key: value for key, value in ticket.items() if key != "gold"})

def one_call(ticket):
    prompt = ticket_text(ticket) + (
        "\nReturn only JSON with decision, policy_id, and refund_usd."
    )
    try:
        return parse_json_loose(llm(prompt, model=MODEL))
    except ValueError:
        return {"decision": "escalate"}

def full_agent(ticket):
    return run_agent(
        ticket_text(ticket),
        standard_tools(),
        model=MODEL,
        system=SYSTEM,
        max_steps=8,
    ).answer

def handle(ticket, lane):
    if lane == "simple_refund":
        return one_call(ticket)
    if lane == "escalate_review":
        return {"decision": "escalate"}
    return full_agent(ticket)

In [ ]:
for ticket in tickets["dev"][:3]:
    lane = route(ticket)
    prediction = handle(ticket, lane)
    print(ticket["ticket_id"], lane, "->", prediction)
    print("known ->", ticket["gold"])


## 2. Prompt chaining

A chain passes one step's output into the next step.

Our chain will:

1. extract two fields from the message;
2. check that the fields are allowed; and
3. give the clean fields to the shop's rule function.

In [ ]:
ALLOWED_CONDITIONS = {"unopened", "opened", "damaged", "defective"}
ALLOWED_ACTIONS = {"refund", "replacement", "store_credit"}

def extract_fields(text):
    prompt = (
        "Extract item_condition and requested_action as JSON. "
        "Return only JSON.\n\n" + text
    )
    return parse_json_loose(llm(prompt, model=MODEL))

def fields_are_safe(fields):
    return (
        isinstance(fields, dict)
        and fields.get("item_condition") in ALLOWED_CONDITIONS
        and fields.get("requested_action") in ALLOWED_ACTIONS
    )

In [ ]:
from shoplab.rules import decide

def run_chain(ticket):
    try:
        fields = extract_fields(ticket["reason_text"])
    except ValueError:
        return {"decision": "escalate", "reason": "no JSON"}

    if not fields_are_safe(fields):
        return {"decision": "escalate", "reason": "fields rejected"}

    clean_ticket = {**ticket, **fields}
    return decide(
        clean_ticket,
        orders[ticket["order_id"]],
        customers[ticket["customer_id"]],
    )

for ticket in tickets["dev"][:2]:
    print(ticket["ticket_id"], run_chain(ticket))
    print("known", ticket["gold"])

The gate is important. If extraction fails, the chain stops. It does not send bad fields into the rule function.

## 3. Parallel work

Independent tickets can run at the same time. We limit the worker count so we do not flood the model provider.

In [ ]:
def run_in_parallel(function, items, workers=4):
    with futures.ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(function, items))

sample = tickets["dev"][:4]
ledger_start = len(LEDGER)
started = time.perf_counter()
predictions = run_in_parallel(full_agent, sample, workers=4)
wall_time = time.perf_counter() - started

call_time = sum(row["seconds"] for row in LEDGER[ledger_start:])
print("tickets:", len(sample))
print("wall time:", round(wall_time, 2), "seconds")
print("sum of call times:", round(call_time, 2), "seconds")

The sum of call times can be larger than wall time because several calls were waiting at the same time.

## 4. Orchestrator and workers

An orchestrator splits a batch into groups. Workers handle each group.

Here the router is the orchestrator's planning step.

In [ ]:
def make_plan(batch):
    lanes = run_in_parallel(route, batch, workers=4)
    plan = {lane: [] for lane in LANES}

    for ticket, lane in zip(batch, lanes):
        plan[lane].append(ticket["ticket_id"])

    return plan

plan = make_plan(tickets["dev"][:6])
for lane, ticket_ids in plan.items():
    print(lane, "->", ticket_ids)

## When a workflow is the better choice

Use a workflow when the steps are known, failures need clear stopping points, or work can be safely split.

Use an agent when the next useful step cannot be known until new information arrives.

## What to remember

1. Routing chooses a handler.
2. Chaining passes clean output to the next step.
3. A gate stops bad data before it travels farther.
4. Parallel work helps when tasks are independent.
5. An orchestrator divides work among workers.
6. Fixed flow is often safer when the path is already known.

## Small practice steps

1. Add a replacement lane to the router.
2. Pass one invalid field through fields_are_safe.
3. Compare two workers with four workers.